# **Audio Analysis Primer**

This notebook introduces core audio concepts used throughout this repo and provides small, runnable examples.

- Sampling rate, channels, normalisation
- Time vs frequency; STFT intuition
- Spectrogram parameters: `n_fft`, `hop_length`, windowing, dB scaling
- Mel scale and MFCC intuition
- Practical tips: clipping, resampling, silence handling



In [ ]:
# Minimal dependency install if needed (ignored in most managed envs)
%pip -q install librosa numpy matplotlib

In [ ]:
import os, sys, logging
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display



In [ ]:
# Adjust sys.path to include the project root (two levels up)
logging.info("Adjusting sys.path to include the project root")
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)
logging.info(f"sys.path adjusted: {sys.path}")



In [ ]:
# Local Imports & Parameters
from config.config import audio_config, output_config
from config.parameters import *
from config.matplotlib_plots import configure_plot, create_custom_colormap
from config.logging import setup_logging


In [ ]:
# Set up logging for this notebook
logging.info("Setting up logging for the notebook")
notebook_path = os.path.join(os.getcwd(), 'audio_primer.ipynb')
setup_logging(notebook_path)
logging.info("Logging set up complete")



## Sampling, Channels, Normalization

- Sampling rate (Hz): samples per second; common: 22050, 44100.
- Channels: mono vs stereo (this repo uses mono loading by default).
- Normalization: keeping amplitudes within [-1, 1] to avoid clipping and ensure comparability.



In [ ]:
def load_audio_by_key(key):
    path = audio_config.get_audio_file(key)
    if not path:
        raise ValueError(f'Unknown audio key: {key}')
    y, sr = librosa.load(path, sr=None, mono=True)
    return y, sr, path

# Example: load sax a3
y, sr, path = load_audio_by_key(AUDIO_FILE_SAX_A3)
time = np.arange(len(y)) / sr

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.plot(time, y, label='Waveform')
configure_plot(ax, title=os.path.basename(path), subtitle='Waveform (Time Domain)')
plt.legend(); plt.tight_layout(); plt.show()



## Time vs Frequency and STFT Intuition

- Time domain shows amplitude over time; frequency domain shows spectral content.
- STFT chops the signal into overlapping windows and runs an FFT per window.
- Trade-off: larger `n_fft` = better frequency resolution but worse time resolution; smaller `hop_length` = denser time sampling but more compute.



In [ ]:
def plot_spectrogram(y, sr, n_fft, hop_length, title_suffix):
    D = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
    fig, ax = plt.subplots(figsize=FIGURE_SIZE)
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)
    img = librosa.display.specshow(
        librosa.amplitude_to_db(D, ref=np.max),
        sr=sr, hop_length=hop_length,
        x_axis='time', y_axis='log', ax=ax
    )
    cbar = fig.colorbar(img, format='%+2.0f dB')
    cbar.ax.set_facecolor(BACKGROUND_COLOR)
    cbar.ax.yaxis.set_tick_params(color=SPINE_COLOR)
    plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=SPINE_COLOR)
    configure_plot(ax, title=f"Spectrogram ({title_suffix})", subtitle=f"n_fft={n_fft}, hop={hop_length}")
    plt.tight_layout(); plt.show()

# Compare different settings
for n_fft, hop in [(512,128),(2048,512),(4096,1024)]:
    plot_spectrogram(y, sr, n_fft, hop, f'n_fft={n_fft}, hop={hop}')



## Mel Scale and MFCC Intuition

- Mel scale warps frequency to approximate human pitch perception.
- Mel-spectrogram: apply a mel filter bank to the magnitude spectrogram.
- MFCCs: take DCT of log-mel energies; often the first 13 coefficients are used.



In [ ]:
# Mel-spectrogram
db_mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64), ref=np.max)
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
img = librosa.display.specshow(db_mel, x_axis='time', y_axis='mel', sr=sr, ax=ax)
fig.colorbar(img, ax=ax)
configure_plot(ax, title=os.path.basename(path), subtitle='Mel-spectrogram (dB)')
plt.tight_layout(); plt.show()

# MFCCs (13)
M = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
img = librosa.display.specshow(M, x_axis='time', ax=ax, cmap=create_custom_colormap())
fig.colorbar(img, ax=ax)
configure_plot(ax, title=os.path.basename(path), subtitle='MFCCs (13)')
plt.tight_layout(); plt.show()



## Practical Tips

- Clipping: inspect waveform; normalize with `y = y / max(|y|)` if needed.
- Resampling: `librosa.resample(y, orig_sr=sr, target_sr)`; ensure anti-aliasing.
- Silence handling: trim leading/trailing silence with `librosa.effects.trim`.
- File formats: WAV preferred for lossless analysis; MP3 may introduce artifacts.



In [ ]:
#  Trimming and resampling

y_trim, idx = librosa.effects.trim(y, top_db=20)
logging.info(f'Trimmed samples: {len(y) - len(y_trim)}')

target_sr = 22050 if sr != 22050 else 16000
y_res = librosa.resample(y_trim, orig_sr=sr, target_sr=target_sr)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.plot(np.arange(len(y_trim))/sr, y_trim, label=f'Trimmed (sr={sr})')
ax.plot(np.arange(len(y_res))/target_sr, y_res, label=f'Resampled (sr={target_sr})', alpha=0.8)
configure_plot(ax, title=os.path.basename(path), subtitle='Trim + Resample Demo')
plt.legend(); plt.tight_layout(); plt.show()

